**© Copyright AIDENTIFY. All rights reserved.**

# Part 1 | Session 05: LangChain 소개 및 기본 실습

---

### 📋 학습 목표

- 🔹 OpenAI API를 직접 호출하여 Chat Completions / role / 스트리밍 원리를 이해합니다
- 🔹 LangChain의 개념과 주요 모듈을 이해합니다
- 🔹 LLM/ChatModel의 기본 사용법을 익힙니다
- 🔹 ChatPromptTemplate으로 프롬프트를 관리합니다
- 🔹 LCEL(LangChain Expression Language)로 파이프라인을 구성합니다
- 🔹 Output Parser로 출력을 구조화합니다
- 🔹 Memory를 활용하여 대화 이력을 관리합니다
- 🔹 간단한 챗봇을 구현합니다

### 📦 필요 라이브러리

```
openai, langchain, langchain-openai, langchain-core, python-dotenv
```

### ⏱️ 예상 소요 시간: 약 100분

---

In [ ]:

# !uv add openai  python-dotenv langchain  langchain-openai langchain-core 



In [6]:
# 필요 라이브러리 설치
# !pip install openai langchain langchain-openai langchain-core python-dotenv

import os
from dotenv import load_dotenv
from openai import OpenAI

# .env 파일에서 환경변수 로드
load_dotenv()

# API Key 확인
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("✅ OpenAI API Key가 설정되었습니다!")
    print(f"   Key 앞 8자: {api_key[:8]}...")
else:
    print("❌ API Key가 설정되지 않았습니다!")
    print("   .env 파일에 OPENAI_API_KEY=sk-... 를 추가해주세요.")

# OpenAI 클라이언트 (Section 1️⃣ 에서 사용)
client = OpenAI(api_key=api_key)
print("\n✅ OpenAI 클라이언트 생성 완료!")

# LangChain 버전 확인 (Section 2️⃣ 이후 사용)
import langchain
print(f"📦 langchain 버전: {langchain.__version__}")

✅ OpenAI API Key가 설정되었습니다!
   Key 앞 8자: sk-proj-...

✅ OpenAI 클라이언트 생성 완료!
📦 langchain 버전: 1.4.3


---

## 🎯 LangChain 소개 및 주요 모듈

### LangChain이란?

LangChain은 LLM 기반 어플리케이션을 쉽게 개발할 수 있도록 도와주는 **프레임워크**입니다.

### LangChain의 핵심 가치

- 🔸 **모듈화**: 각 기능을 독립된 모듈로 제공
- 🔸 **조합 가능**: 모듈을 파이프라인으로 연결
- 🔸 **확장 가능**: 다양한 LLM, 벡터DB, 도구 지원
- 🔸 **추상화**: 복잡한 로직을 간단한 인터페이스로

### 주요 모듈 구조

```
LangChain 생태계
  │
  ├── langchain-core       : 핵심 추상화 (LCEL, 메시지, 프롬프트)
  ├── langchain            : 체인, 에이전트, 메모리
  ├── langchain-openai     : OpenAI 통합
  ├── langchain-community  : 커뮤니티 통합
  ├── langgraph            : 에이전트 워크플로우
  └── langsmith            : 디버깅/모니터링
```

### 핵심 구성요소

| 구성요소 | 설명 | 예시 |
|---------|------|------|
| Model | LLM 모델 래퍼 | ChatOpenAI, ChatOllama |
| Prompt | 프롬프트 템플릿 | ChatPromptTemplate |
| Output Parser | 출력 파싱 | StrOutputParser, JsonOutputParser |
| Chain | 모듈 조합 | LCEL 파이프라인 |
| Memory | 대화 이력 관리 | ConversationBufferMemory |
| Retriever | 문서 검색 | VectorStoreRetriever |
| Agent | 도구 활용 자율 행동 | ReAct Agent |

---

---

## 1️⃣ OpenAI API 직접 호출 (LangChain 없이)

LangChain은 결국 OpenAI/Anthropic 등의 API를 **추상화**한 프레임워크입니다. 추상화 계층을 이해하려면 그 아래에 있는 **원본 API**가 어떻게 동작하는지 먼저 보는 것이 도움이 됩니다.

### Chat Completions API 구조

OpenAI의 가장 핵심 엔드포인트는 `chat.completions.create()` 입니다. 핵심 입력은 `messages` 리스트로, 각 메시지는 **role**(역할)과 **content**(내용)를 가집니다.

| Role | 설명 | 사용 시점 |
|------|------|----------|
| `system` | 모델의 역할/지침 설정 | 첫 메시지로 1회 |
| `user` | 사용자 입력 | 매 턴 |
| `assistant` | 모델의 이전 응답 | 멀티턴 대화 시 이력 |

### 호출 흐름

```python
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "당신은 친절한 한국어 도우미입니다."},
        {"role": "user",   "content": "LLM 파인튜닝이 뭐야?"}
    ],
    temperature=0.7,
    max_tokens=300,
)
answer = response.choices[0].message.content
```

> 💡 이 섹션에서 본 `messages` / `temperature` / `stream` 개념은 모두 LangChain의 `ChatOpenAI`로 그대로 전달됩니다. 다음 섹션부터는 LangChain이 이를 어떻게 추상화하는지 봅니다.

---

In [7]:
# Chat Completions API 기본 호출
print("=" * 60)
print("💬 OpenAI Chat Completions API — 기본 호출")
print("=" * 60)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "LLM 파인튜닝이 뭔지 3줄로 설명해줘."}
    ],
    temperature=0.7,
    max_tokens=300,
)

# 응답 내용
answer = response.choices[0].message.content
print(f"\n🤖 모델 응답:\n{answer}")

# 응답 메타데이터
print(f"\n📊 응답 메타데이터:")
print(f"   모델: {response.model}")
print(f"   입력 토큰: {response.usage.prompt_tokens}")
print(f"   출력 토큰: {response.usage.completion_tokens}")
print(f"   총 토큰: {response.usage.total_tokens}")
print(f"   종료 이유: {response.choices[0].finish_reason}")


💬 OpenAI Chat Completions API — 기본 호출



🤖 모델 응답:
LLM 파인튜닝(대규모 언어 모델 파인튜닝)은 사전 훈련된 언어 모델을 특정 작업이나 도메인에 맞게 조정하는 과정입니다. 이 과정에서는 적은 양의 데이터로 모델의 성능을 개선하고, 특정한 요구사항에 맞춘 결과를 생성할 수 있도록 합니다. 일반적으로, 파인튜닝은 모델의 가중치를 미세하게 조정하여 더 나은 예측을 하도록 합니다.

📊 응답 메타데이터:
   모델: gpt-4o-mini-2024-07-18
   입력 토큰: 25
   출력 토큰: 104
   총 토큰: 129
   종료 이유: stop


In [8]:
# 시스템 프롬프트로 페르소나/형식 지정
print("=" * 60)
print("🎭 시스템 프롬프트 활용")
print("=" * 60)

# 예시 1: 한영 번역가
print("\n--- 예시 1: 한영 전문 번역가 ---")
res1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": (
            "당신은 한영 전문 번역가입니다. "
            "한국어가 입력되면 자연스러운 영어로 번역하고, 결과만 출력하세요."
        )},
        {"role": "user", "content": "파인튜닝은 사전 학습된 모델을 특정 도메인에 맞게 추가 학습하는 과정입니다."},
    ],
    temperature=0.3,
    max_tokens=200,
)
print(f"🤖 번역 결과:\n{res1.choices[0].message.content}")

# 예시 2: 같은 질문을 다른 system 프롬프트로
print("\n--- 예시 2: system 프롬프트의 영향 ---")
question = "AI에 대해 알려줘."

# 페르소나 A: 초등학생 눈높이
res_a = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "당신은 초등학생에게 설명하는 선생님입니다. 쉬운 단어와 비유를 사용하세요."},
        {"role": "user", "content": question},
    ],
    temperature=0.5,
    max_tokens=150,
)
print(f"\n[페르소나 A — 초등학생 눈높이]\n{res_a.choices[0].message.content}")

# 페르소나 B: 대학원생 수준
res_b = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "당신은 대학원에서 AI를 가르치는 교수입니다. 학술적이고 정확한 용어를 사용하세요."},
        {"role": "user", "content": question},
    ],
    temperature=0.5,
    max_tokens=200,
)
print(f"\n[페르소나 B — 대학원생 수준]\n{res_b.choices[0].message.content}")

print("\n✅ 같은 질문도 system 프롬프트에 따라 답변 스타일이 완전히 달라집니다.")


🎭 시스템 프롬프트 활용

--- 예시 1: 한영 전문 번역가 ---


🤖 번역 결과:
Fine-tuning is the process of further training a pre-trained model to adapt it to a specific domain.

--- 예시 2: system 프롬프트의 영향 ---

[페르소나 A — 초등학생 눈높이]
좋아! AI는 "인공지능"의 줄임말이야. 쉽게 말하면, AI는 컴퓨터나 기계가 사람처럼 생각하고 배우는 능력을 가지게 하는 기술이야.

예를 들어, 우리가 친구와 대화할 때 친구는 우리의 말을 듣고 이해하고, 그에 맞게 대답하지? AI도 비슷하게, 많은 정보를 배우고, 그 정보를 바탕으로 질문에 대답하거나 문제를 해결할 수 있어.

AI는 마치 똑똑한 로봇 같아. 우리가 게임을 할 때, 게임 속의 캐릭터가 똑똑하게 움직이고 반응하는 것도 AI 덕분이야. 또, 우리가 사용하는

[페르소나 B — 대학원생 수준]
인공지능(AI, Artificial Intelligence)은 인간의 인지적 기능을 모방하여 문제를 해결하거나 작업을 수행할 수 있는 시스템이나 기계를 지칭합니다. AI는 여러 하위 분야로 나눌 수 있으며, 이들 각각은 특정한 문제를 해결하기 위한 다양한 접근 방식을 포함합니다.

1. **기계 학습(Machine Learning)**: AI의 한 분야로, 데이터로부터 학습하여 예측이나 결정을 내리는 알고리즘을 개발하는 데 중점을 둡니다. 기계 학습은 지도 학습, 비지도 학습, 강화 학습 등으로 further 세분화될 수 있습니다.

2. **신경망(Neural Networks)**: 생물학적 신경망에서 영감을 받아 설계된 구조로, 다층의 노드(뉴런)로 구성되어 있습니다. 딥러닝(Deep Learning)은 이러한 신경망의 심화된 형태로

✅ 같은 질문도 system 프롬프트에 따라 답변 스타일이 완전히 달라집니다.


In [9]:
# 스트리밍 응답: 토큰이 생성되는 대로 받기
import time

print("=" * 60)
print("🌊 스트리밍(Streaming) 응답")
print("=" * 60)

print("\n📝 질문: LLM 파인튜닝의 3단계를 설명해주세요.")
print("\n🤖 응답 (실시간 출력):")
print("-" * 50)

start = time.time()
first_token_time = None
full_response = ""

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "한국어로 간결하게 답변하세요."},
        {"role": "user",   "content": "LLM 파인튜닝의 3단계를 설명해주세요."},
    ],
    temperature=0.7,
    max_tokens=500,
    stream=True,   # ⭐ 핵심: 스트리밍 활성화
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta is not None:
        if first_token_time is None:
            first_token_time = time.time()
        print(delta, end="", flush=True)
        full_response += delta

end = time.time()
print("\n" + "-" * 50)
print(f"\n⏱️ 첫 토큰까지: {first_token_time - start:.2f}초")
print(f"⏱️ 전체 완료: {end - start:.2f}초")
print(f"📊 총 글자 수: {len(full_response)}")
print("\n💡 스트리밍은 사용자가 첫 글자를 빨리 보게 해 UX 체감 속도를 크게 높입니다.")


🌊 스트리밍(Streaming) 응답

📝 질문: LLM 파인튜닝의 3단계를 설명해주세요.

🤖 응답 (실시간 출력):
--------------------------------------------------


LLM(대규모 언어 모델) 파인튜닝의 3단계는 다음과 같습니다.

1. **데이터 준비**: 특정 작업이나 도메인에 맞는 데이터셋을 수집하고, 이 데이터를 전처리하여 모델이 학습할 수 있는 형식으로 변환합니다.

2. **모델 설정**: 파인튜닝할 기존의 LLM을 선택하고, 하이퍼파라미터(학습률, 배치 크기 등)를 설정합니다. 이 단계에서는 초기 가중치를 로드하고, 필요한 경우 모델 구조를 조정할 수 있습니다.

3. **학습 및 평가**: 준비된 데이터셋을 사용하여 모델을 학습시키고, 검증 데이터셋으로 성능을 평가합니다. 성능이 만족스럽지 않으면 하이퍼파라미터를 조정하거나 추가적인 데이터로 재학습을 진행할 수 있습니다.
--------------------------------------------------

⏱️ 첫 토큰까지: 0.59초
⏱️ 전체 완료: 2.45초
📊 총 글자 수: 351

💡 스트리밍은 사용자가 첫 글자를 빨리 보게 해 UX 체감 속도를 크게 높입니다.


---

## 2️⃣ LangChain LLM/ChatModel 기본 사용

이제 위에서 본 OpenAI API 호출이 LangChain에서 어떻게 표현되는지 봅니다. LangChain의 `ChatOpenAI`는 내부적으로 `client.chat.completions.create()` 를 호출하지만, 메시지 객체와 체이닝을 위한 인터페이스를 추가로 제공합니다.

### ChatModel vs LLM

| 구분 | ChatModel | LLM |
|------|-----------|-----|
| 입력 | 메시지 리스트 | 문자열 |
| 출력 | AIMessage 객체 | 문자열 |
| 예시 | ChatOpenAI | OpenAI (deprecated) |

최신 LangChain에서는 **ChatModel**을 사용하는 것이 권장됩니다.

---

In [10]:
# ChatOpenAI 기본 사용
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

print("=" * 60)
print("🤖 ChatOpenAI 기본 사용")
print("=" * 60)

# ChatModel 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    max_tokens=300,
)

print("✅ ChatOpenAI 모델 생성 완료!")
print(f"   모델: {llm.model_name}")
print(f"   Temperature: {llm.temperature}")

# 방법 1: invoke() - 가장 기본적인 호출
print("\n--- 방법 1: invoke() ---")
response = llm.invoke("LangChain이 뭔지 한 문장으로 설명해줘.")
print(f"🤖 응답 타입: {type(response).__name__}")
print(f"🤖 응답: {response.content}")

🤖 ChatOpenAI 기본 사용
✅ ChatOpenAI 모델 생성 완료!
   모델: gpt-4o-mini
   Temperature: 0.7

--- 방법 1: invoke() ---
🤖 응답 타입: AIMessage
🤖 응답: LangChain은 다양한 자연어 처리(NLP) 모델과 도구를 통합하여 언어 기반 애플리케이션을 개발할 수 있도록 돕는 프레임워크입니다.


In [11]:
# 메시지 객체 사용
print("=" * 60)
print("📨 메시지 객체 사용")
print("=" * 60)

# 방법 2: 메시지 리스트로 호출
messages = [
    SystemMessage(content="당신은 파이썬 전문가입니다. 한국어로 간결하게 답변하세요."),
    HumanMessage(content="리스트 컴프리헨션이 뭐야?"),
]

response = llm.invoke(messages)
print(f"\n🤖 응답:\n{response.content}")

# 응답 메타데이터
print(f"\n📊 메타데이터:")
print(f"   토큰 사용: {response.usage_metadata}")
print(f"   응답 ID: {response.id}")

📨 메시지 객체 사용



🤖 응답:
리스트 컴프리헨션(List Comprehension)은 파이썬에서 리스트를 간결하게 생성하는 방법입니다. 기존 리스트를 기반으로 새로운 리스트를 만들 수 있으며, 조건문도 포함할 수 있습니다. 

예를 들어, 0부터 9까지의 제곱수를 담은 리스트를 만들고 싶다면 다음과 같이 사용할 수 있습니다:

```python
squares = [x**2 for x in range(10)]
```

또한 조건을 추가할 수도 있습니다:

```python
even_squares = [x**2 for x in range(10) if x % 2 == 0]
```

이렇게 하면 짝수에 대한 제곱수만 포함된 리스트를 생성할 수 있습니다.

📊 메타데이터:
   토큰 사용: {'input_tokens': 43, 'output_tokens': 158, 'total_tokens': 201, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
   응답 ID: lc_run--01a0f94d-de2c-7570-8121-dfb7682a1529-0


In [12]:
# 스트리밍 사용
print("=" * 60)
print("🌊 LangChain 스트리밍")
print("=" * 60)

print("\n🤖 스트리밍 응답:")
print("-" * 40)

# stream() 메서드로 스트리밍
for chunk in llm.stream("파인튜닝의 장점을 3가지 알려줘."):
    print(chunk.content, end="", flush=True)

print("\n" + "-" * 40)
print("\n✅ stream() 메서드로 간편하게 스트리밍 가능!")

🌊 LangChain 스트리밍

🤖 스트리밍 응답:
----------------------------------------


파인튜닝(fine-tuning)은 사전 학습된 모델을 특정 작업에 맞게 조정하는 과정입니다. 이 과정의 주요 장점은 다음과 같습니다:

1. **효율성**: 파인튜닝은 사전 학습된 모델을 기반으로 하므로, 처음부터 모든 것을 학습하는 것보다 훨씬 적은 데이터와 시간으로 원하는 성능을 달성할 수 있습니다. 이는 특히 데이터가 적거나 학습 시간이 제한된 상황에서 유리합니다.

2. **전이 학습**: 사전 학습된 모델은 다양한 데이터에서 학습한 일반적인 패턴과 특성을 가지고 있습니다. 이를 활용함으로써, 특정 작업에 대한 데이터가 부족하더라도 모델이 이미 학습한 정보를 바탕으로 더 나은 성능을 발휘할 수 있습니다.

3. **성능 향상**: 파인튜닝을 통해 특정 도메인이나 작업에 맞게 모델을 최적화함으로써, 기존의 사전 학습된 모델보다 더 높은 정확도와 성능을 얻을 수 있습니다. 이는 특히 복잡한 문제를 해결하는 데 큰 도움이 됩니다.

이러한 장점 덕분에 파인튜닝은 자연어 처리(NLP), 이미지 인식, 음성 인식 등 다양한 분야에서 널리 사용되고 있습니다.
----------------------------------------

✅ stream() 메서드로 간편하게 스트리밍 가능!


---

## 3️⃣ 프롬프트 템플릿 (ChatPromptTemplate)

프롬프트를 **재사용 가능한 템플릿**으로 관리합니다.

### 왜 프롬프트 템플릿이 필요한가?

- 🔸 프롬프트의 **재사용성** 향상
- 🔸 **변수 치환**으로 동적 프롬프트 생성
- 🔸 프롬프트 **버전 관리** 용이
- 🔸 **일관된 형식** 유지

---

In [13]:
# ChatPromptTemplate 기본 사용
from langchain_core.prompts import ChatPromptTemplate

print("=" * 60)
print("📝 ChatPromptTemplate 기본 사용")
print("=" * 60)

# 방법 1: from_messages로 생성
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다. {language}로 답변하세요."),
    ("human", "{question}"),
])

print("📌 템플릿 변수:")
print(f"   {prompt.input_variables}")

# 변수 치환하여 메시지 생성
messages = prompt.format_messages(
    role="머신러닝",
    language="한국어",
    question="오버피팅을 방지하는 방법은?"
)

print(f"\n📨 생성된 메시지:")
for msg in messages:
    print(f"   [{msg.type}] {msg.content}")

# LLM 호출
response = llm.invoke(messages)
print(f"\n🤖 응답:\n{response.content}")

📝 ChatPromptTemplate 기본 사용
📌 템플릿 변수:
   ['language', 'question', 'role']

📨 생성된 메시지:
   [system] 당신은 머신러닝 전문가입니다. 한국어로 답변하세요.
   [human] 오버피팅을 방지하는 방법은?

🤖 응답:
오버피팅은 모델이 훈련 데이터에 지나치게 맞춰져 새로운 데이터에 대한 일반화 능력이 떨어지는 현상입니다. 오버피팅을 방지하기 위한 여러 가지 방법이 있습니다:

1. **더 많은 데이터 수집**: 훈련 데이터의 양을 늘리면 모델이 더 일반화된 패턴을 학습할 수 있습니다.

2. **데이터 증강**: 기존 데이터에 변형을 가해 새로운 데이터를 생성하여 데이터 양을 늘리는 방법입니다. 예를 들어, 이미지 데이터의 경우 회전, 확대, 색상 변화 등을 적용할 수 있습니다.

3. **정규화 기법**:
   - **L1 & L2 정규화**: 모델의 복잡도를 줄이기 위해 가중치에 대한 패널티를 추가합니다.
   - **드롭아웃**: 신경망의 특정 뉴런을 랜덤하게 제거하여 과적합을 방지합니다.

4. **모델 단순화**: 복잡한 모델 대신 더 간단한 모델을 선택하거나, 층의 수나 뉴런의 수를 줄여 모델의 복잡성을 낮추는 방법입니다.

5. **교차 검증**: 데이터를 여러 개의 부분으로 나누어 모델을 평가함으로써, 모델의 일반화 성능을 확인할 수 있습니다.

6. **조기 종료(Early St


In [14]:
# 다양한 프롬프트 템플릿 예시
print("=" * 60)
print("📝 실용적인 프롬프트 템플릿 예시")
print("=" * 60)

# 번역 템플릿
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "{source_lang}를 {target_lang}로 번역하세요. 번역 결과만 출력하세요."),
    ("human", "{text}"),
])

# 요약 템플릿
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 텍스트를 {num_sentences}문장으로 요약하세요."),
    ("human", "{text}"),
])

# 번역 실행
print("\n--- 번역 ---")
msg = translate_prompt.format_messages(
    source_lang="한국어",
    target_lang="영어",
    text="파인튜닝은 사전 학습된 모델을 특정 작업에 맞게 추가 학습하는 기법입니다."
)
response = llm.invoke(msg)
print(f"🤖 {response.content}")

# 요약 실행
print("\n--- 요약 ---")
msg = summary_prompt.format_messages(
    num_sentences=2,
    text="""트랜스포머는 2017년 구글에서 발표한 신경망 아키텍처입니다. 
    기존 RNN/LSTM의 순차적 처리 한계를 극복하고, 셀프 어텐션 메커니즘을 통해 
    입력 시퀀스의 모든 위치를 동시에 참조할 수 있습니다. 이를 통해 병렬 처리가 가능해져 
    학습 속도가 크게 향상되었으며, GPT, BERT, T5 등 현대 LLM의 기반이 되었습니다."""
)
response = llm.invoke(msg)
print(f"🤖 {response.content}")

print("\n✅ 템플릿을 미리 정의하면 일관된 프롬프트를 재사용할 수 있습니다!")

📝 실용적인 프롬프트 템플릿 예시

--- 번역 ---


🤖 Fine-tuning is a technique that involves additional training of a pre-trained model to adapt it to a specific task.

--- 요약 ---
🤖 트랜스포머는 2017년 구글이 발표한 신경망 아키텍처로, RNN/LSTM의 한계를 극복하고 셀프 어텐션 메커니즘을 통해 입력 시퀀스를 동시에 참조할 수 있게 합니다. 이로 인해 병렬 처리가 가능해져 학습 속도가 향상되었으며, 현대 LLM(GPT, BERT, T5)의 기반이 되었습니다.

✅ 템플릿을 미리 정의하면 일관된 프롬프트를 재사용할 수 있습니다!


---

## 4️⃣ LCEL (LangChain Expression Language) 파이프라인

LCEL은 LangChain의 **파이프라인 구성 문법**입니다. `|` (파이프) 연산자로 모듈을 연결합니다.

### LCEL의 핵심 개념

```python
chain = prompt | model | output_parser
#       입력     →    처리     →    출력 파싱
```

### LCEL의 장점

- 🔸 **직관적 구문**: `|`로 데이터 흐름을 표현
- 🔸 **자동 스트리밍**: chain.stream()으로 자동 지원
- 🔸 **비동기 지원**: chain.ainvoke()로 비동기 실행
- 🔸 **배치 처리**: chain.batch()로 대량 처리
- 🔸 **재시도/폴백**: with_retry(), with_fallbacks()

---

In [15]:
# LCEL 기본 파이프라인
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

print("=" * 60)
print("🔗 LCEL 기본 파이프라인")
print("=" * 60)

# 파이프라인 구성: 프롬프트 → 모델 → 출력 파서
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {topic} 전문가입니다. 한국어로 간결하게 답변하세요."),
    ("human", "{question}"),
])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
output_parser = StrOutputParser()

# LCEL 체인 생성 ( | 연산자로 연결)
chain = prompt | model | output_parser

print("✅ LCEL 체인 생성 완료!")
print(f"   체인 구조: prompt → model → output_parser")

# invoke() 호출
result = chain.invoke({
    "topic": "딥러닝",
    "question": "CNN과 RNN의 차이점을 설명해주세요."
})

print(f"\n📝 입력: topic='딥러닝', question='CNN과 RNN의 차이점을 설명해주세요.'")
print(f"\n🤖 출력 (문자열):")
print(result)
print(f"\n📊 출력 타입: {type(result).__name__}")
print("💡 StrOutputParser가 AIMessage를 문자열로 변환했습니다!")

🔗 LCEL 기본 파이프라인
✅ LCEL 체인 생성 완료!
   체인 구조: prompt → model → output_parser



📝 입력: topic='딥러닝', question='CNN과 RNN의 차이점을 설명해주세요.'

🤖 출력 (문자열):
CNN(합성곱 신경망)과 RNN(순환 신경망)의 주요 차이점은 다음과 같습니다:

1. **구조**:
   - CNN: 주로 이미지 처리에 사용되며, 합성곱 층과 풀링 층이 포함되어 있어 특징 추출에 효과적입니다.
   - RNN: 시퀀스 데이터(예: 텍스트, 시간 시계열)를 처리하는 데 적합하며, 이전 정보를 기억하기 위해 순환 구조를 가집니다.

2. **입력 데이터 형식**:
   - CNN: 2D 또는 3D 형태의 데이터(예: 이미지)를 처리합니다.
   - RNN: 1D 시퀀스 데이터를 처리하며, 시간적 의존성을 모델링합니다.

3. **주요 응용 분야**:
   - CNN: 이미지 분류, 객체 탐지, 영상 분석 등에 사용됩니다.
   - RNN: 자연어 처리, 음성 인식, 시계열 예측 등에 사용됩니다.

이러한 차이로 인해 각 네트워크는 특정 유형의 데이터와 작업에 더 적합합니다.

📊 출력 타입: TextAccessor
💡 StrOutputParser가 AIMessage를 문자열로 변환했습니다!


In [16]:
# LCEL 스트리밍과 배치 처리
print("=" * 60)
print("🔗 LCEL 스트리밍 & 배치 처리")
print("=" * 60)

# 스트리밍
print("\n--- 스트리밍 ---")
print("🤖 ", end="")
for chunk in chain.stream({"topic": "파이썬", "question": "데코레이터를 간단히 설명해줘."}):
    print(chunk, end="", flush=True)
print()

# 배치 처리
print("\n--- 배치 처리 ---")
questions = [
    {"topic": "파이썬", "question": "GIL이 뭐야?"},
    {"topic": "딥러닝", "question": "드롭아웃이 뭐야?"},
    {"topic": "데이터", "question": "정규화가 뭐야?"},
]

results = chain.batch(questions)

for q, r in zip(questions, results):
    print(f"\n📝 [{q['topic']}] {q['question']}")
    print(f"🤖 {r[:100]}...")

print(f"\n✅ batch()로 {len(questions)}개의 질문을 한 번에 처리했습니다!")

🔗 LCEL 스트리밍 & 배치 처리

--- 스트리밍 ---
🤖 

데코레이터는 함수나 메서드의 기능을 확장하거나 수정하는 함수입니다. 다른 함수를 인자로 받아 새로운 함수를 반환하며, 주로 코드 재사용과 가독성을 높이는 데 사용됩니다. 예를 들어, 함수 실행 전후에 특정 작업을 수행할 수 있습니다. 사용법은 `@데코레이터이름` 형태로 간단하게 적용할 수 있습니다.

--- 배치 처리 ---

📝 [파이썬] GIL이 뭐야?
🤖 GIL(전역 인터프리터 잠금, Global Interpreter Lock)은 CPython에서 여러 스레드가 동시에 실행될 때, 한 번에 하나의 스레드만 파이썬 바이트코드를 실행할...

📝 [딥러닝] 드롭아웃이 뭐야?
🤖 드롭아웃(Dropout)은 신경망의 과적합(overfitting)을 방지하기 위해 사용되는 정규화 기법입니다. 학습 과정에서 무작위로 일부 뉴런을 "드롭"하여 비활성화함으로써, 모...

📝 [데이터] 정규화가 뭐야?
🤖 정규화는 데이터베이스 설계 과정에서 데이터 중복을 최소화하고, 데이터 무결성을 향상시키기 위해 데이터 구조를 정리하는 방법입니다. 일반적으로 여러 단계(정규형)를 통해 진행되며, ...

✅ batch()로 3개의 질문을 한 번에 처리했습니다!


In [17]:
# ainvoke vs batch 비교
import asyncio
import time

print("=" * 60)
print("⚡ ainvoke vs batch 비교")
print("=" * 60)

questions = [
    {"topic": "파이썬", "question": "GIL이 뭐야?"},
    {"topic": "딥러닝", "question": "드롭아웃이 뭐야?"},
    {"topic": "데이터", "question": "정규화가 뭐야?"},
]

# ===== 1. invoke 순차 처리 =====
print("\n1️⃣ invoke() — 순차 처리")
start = time.time()
results_sync = []
for q in questions:
    results_sync.append(chain.invoke(q))
sync_time = time.time() - start
print(f"   ⏱️ {sync_time:.2f}초")

# ===== 2. batch 동시 처리 =====
print("\n2️⃣ batch() — 스레드 동시 처리")
start = time.time()
results_batch = chain.batch(questions)
batch_time = time.time() - start
print(f"   ⏱️ {batch_time:.2f}초")

# ===== 3. ainvoke 비동기 동시 처리 =====
print("\n3️⃣ ainvoke() — 비동기 동시 처리")

async def run_async():
    start = time.time()
    results = await asyncio.gather(
        chain.ainvoke(questions[0]),
        chain.ainvoke(questions[1]),
        chain.ainvoke(questions[2]),
    )
    elapsed = time.time() - start
    return results, elapsed

results_async, async_time = await run_async()

# ===== 결과 비교 =====
print(f"   ⏱️ {async_time:.2f}초")
print("\n" + "=" * 60)
print("📊 시간 비교")
print("=" * 60)
print(f"   invoke  (순차):  {sync_time:.2f}초")
print(f"   batch   (스레드): {batch_time:.2f}초")
print(f"   ainvoke (비동기): {async_time:.2f}초")
print(f"\n💡 batch와 ainvoke는 비슷한 속도, invoke 순차 처리는 느림!")
print("💡 Jupyter는 이미 이벤트 루프가 있어서 await를 바로 사용할 수 있습니다.")

⚡ ainvoke vs batch 비교

1️⃣ invoke() — 순차 처리


   ⏱️ 3.73초

2️⃣ batch() — 스레드 동시 처리
   ⏱️ 1.44초

3️⃣ ainvoke() — 비동기 동시 처리
   ⏱️ 2.03초

📊 시간 비교
   invoke  (순차):  3.73초
   batch   (스레드): 1.44초
   ainvoke (비동기): 2.03초

💡 batch와 ainvoke는 비슷한 속도, invoke 순차 처리는 느림!
💡 Jupyter는 이미 이벤트 루프가 있어서 await를 바로 사용할 수 있습니다.


In [18]:
# 체인 연결: 다단계 파이프라인
print("=" * 60)
print("🔗 다단계 LCEL 파이프라인")
print("=" * 60)

from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# 1단계: 질문에 대한 답변 생성
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 질문에 한국어로 상세히 답변하세요."),
    ("human", "{question}"),
])

# 2단계: 답변을 요약
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 텍스트를 핵심만 2줄로 요약하세요."),
    ("human", "{text}"),
])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)
parser = StrOutputParser()

# 1단계 체인
answer_chain = answer_prompt | model | parser


# 2단계 체인 (1단계 결과를 입력으로)
full_chain = (
    answer_chain  # 1단계: 답변 생성
    | RunnableLambda(lambda x: {"text": x})  # 결과를 dict로 변환
    | summary_prompt  # 2단계: 요약 프롬프트
    | model  # 2단계: 모델 호출
    | parser  # 2단계: 문자열 파싱
)

# 실행
question = "트랜스포머의 셀프 어텐션 메커니즘이 뭐야?"
print(f"\n📝 질문: {question}")

# 1단계 결과
print(f"\n--- 1단계: 상세 답변 ---")
answer = answer_chain.invoke({"question": question})
print(f"🤖 {answer}")

# 2단계 결과 (전체 파이프라인)
print(f"\n--- 2단계: 요약 ---")
summary = full_chain.invoke({"question": question})
print(f"🤖 {summary}")

print("\n✅ 다단계 파이프라인으로 '답변 생성 → 요약'을 자동화했습니다!")

🔗 다단계 LCEL 파이프라인

📝 질문: 트랜스포머의 셀프 어텐션 메커니즘이 뭐야?

--- 1단계: 상세 답변 ---


🤖 트랜스포머의 셀프 어텐션(self-attention) 메커니즘은 자연어 처리(NLP) 분야에서 중요한 역할을 하는 기술로, 입력 데이터의 각 요소가 서로 어떻게 관련되어 있는지를 이해하고 표현하는 방법입니다. 이 메커니즘은 트랜스포머 모델의 핵심 구성 요소 중 하나로, 특히 문맥을 고려하여 단어 간의 관계를 파악하는 데 유용합니다.

셀프 어텐션은 다음과 같은 과정을 거칩니다:

1. **입력 벡터 변환**: 입력 문장의 각 단어는 고차원 벡터로 표현됩니다. 이 벡터들은 트랜스포머의 입력으로 사용됩니다.

2. **쿼리, 키, 값 생성**: 각 입력 벡터는 세 가지 다른 벡터로 변환됩니다. 이를 각각 쿼리(query), 키(key), 값(value)라고 합니다. 이 변환은 학습 가능한 가중치 행렬을 사용하여 이루어집니다.

3. **어텐션 스코어 계산**: 각 쿼리 벡터와 모든 키 벡터 간의 내적(dot product)을 계산하여 어텐션 스코어를 생성합니다. 이 스코어는 특정 단어가 다른 단어에 얼마나 주의를 기울여야 하는지를 나타냅니다.

4. **스코어 정규화**: 계산된 어텐션 스코어는 소프트맥스(softmax) 함수를 통해 정규화됩니다. 이 과정을 통해 각 단어의 주의도(attention weight)를 얻습니다. 주의도는 0과 1 사이의 값으로 표현되며, 모든 주의도의 합은 1이 됩니다.

5. **가중합**: 최종적으로 각 값 벡터(value)에 해당하는 주의도를 곱하여 가중합을 구합니다. 이렇게 얻어진 벡터는 입력 단어의 문맥 정보를 반영한 새로운 표현이 됩니다.

셀프 어텐션의 주요 장점은 다음과 같습니다:

- **병렬 처리**: 셀프 어텐션은 입력의 모든 단어를 동시에 처리할 수 있어, 순차적으로 처리해야 하는 RNN보다 훨씬 빠릅니다.
- **장기 의존성**: 셀프 어텐션은 입력의 모든 단어 간의 관계를 고려할 수 있어, 문맥을 더 잘 이해할 수 있습니다.
- **유연한 길이**: 입력의 길이에 상관없이 처리할 수 있어 다양한 길이의 문

---

## 5️⃣ Output Parser 활용

Output Parser는 LLM 출력을 **원하는 형태**로 변환합니다.

### 주요 Output Parser

| Parser | 설명 | 출력 타입 |
|--------|------|----------|
| StrOutputParser | 문자열로 변환 | str |
| JsonOutputParser | JSON으로 파싱 | dict |
| CommaSeparatedListOutputParser | 쉼표 구분 리스트 | list |
| PydanticOutputParser | Pydantic 모델로 파싱 | Pydantic 객체 |

---

In [19]:
# 다양한 Output Parser 활용
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, CommaSeparatedListOutputParser

print("=" * 60)
print("📤 Output Parser 활용")
print("=" * 60)

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 1. CommaSeparatedListOutputParser
print("\n--- 1. CommaSeparatedListOutputParser ---")
list_parser = CommaSeparatedListOutputParser()

list_prompt = ChatPromptTemplate.from_messages([
    ("system", "답변을 쉼표로 구분된 리스트로 작성하세요. {format_instructions}"),
    ("human", "{question}"),
])

list_chain = list_prompt | model | list_parser

result = list_chain.invoke({
    "question": "인기 있는 딥러닝 프레임워크 5가지를 알려줘",
    "format_instructions": list_parser.get_format_instructions(),
})

print(f"📊 결과 타입: {type(result).__name__}")
print(f"📊 결과: {result}")
print(f"📊 첫 번째 항목: {result[0]}")

# 2. JsonOutputParser
print("\n--- 2. JsonOutputParser ---")
json_parser = JsonOutputParser()

json_prompt = ChatPromptTemplate.from_messages([
    ("system", """다음 형식의 JSON으로 응답하세요:
{{"name": "모델명", "organization": "개발사", "parameters": "파라미터수", "year": 연도}}
{format_instructions}"""),
    ("human", "{question}"),
])

json_chain = json_prompt | model | json_parser

result = json_chain.invoke({
    "question": "GPT-4에 대해 알려줘",
    "format_instructions": json_parser.get_format_instructions(),
})

print(f"📊 결과 타입: {type(result).__name__}")
print(f"📊 결과: {result}")
print(f"📊 모델명: {result.get('name')}")

print("\n✅ Output Parser로 LLM 출력을 프로그래밍에 바로 활용할 수 있습니다!")

📤 Output Parser 활용

--- 1. CommaSeparatedListOutputParser ---
📊 결과 타입: list
📊 결과: ['TensorFlow', 'PyTorch', 'Keras', 'MXNet', 'Caffe']
📊 첫 번째 항목: TensorFlow

--- 2. JsonOutputParser ---
📊 결과 타입: dict
📊 결과: {'name': 'GPT-4', 'organization': 'OpenAI', 'parameters': '수천억', 'year': 2023}
📊 모델명: GPT-4

✅ Output Parser로 LLM 출력을 프로그래밍에 바로 활용할 수 있습니다!


---

## 6️⃣ Memory와 대화 이력 관리

LLM API는 기본적으로 **상태를 저장하지 않습니다(stateless)**.

LangChain의 Memory 모듈을 사용하면 **대화 이력을 자동으로 관리**할 수 있습니다.

### 주요 Memory 유형

| Memory | 설명 | 사용 사례 |
|--------|------|----------|
| ConversationBufferMemory | 전체 대화 저장 | 짧은 대화 |
| ConversationBufferWindowMemory | 최근 N턴만 저장 | 일반적인 챗봇 |
| ConversationSummaryMemory | 대화를 요약하여 저장 | 긴 대화 |
| ConversationTokenBufferMemory | 토큰 수 기반 관리 | 비용 최적화 |

### 최신 LangChain 방식: 메시지 히스토리 직접 관리

최신 LangChain에서는 `RunnableWithMessageHistory`나 직접 메시지 리스트를 관리하는 방식이 권장됩니다.

---

In [20]:
# 메시지 히스토리 직접 관리 방식 (최신 LangChain 권장)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

print("=" * 60)
print("💾 메시지 히스토리를 활용한 대화 관리")
print("=" * 60)

# MessagesPlaceholder를 사용한 프롬프트
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다. 한국어로 답변하세요."),
    MessagesPlaceholder(variable_name="history"),  # 대화 이력이 여기에 삽입
    ("human", "{input}"),
])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
parser = StrOutputParser()
chain = prompt | model | parser

# 대화 이력 저장소
chat_history = []

def chat_with_history(user_input):
    """대화 이력을 관리하면서 채팅"""
    # 체인 실행
    response = chain.invoke({
        "history": chat_history,
        "input": user_input,
    })
    
    # 대화 이력에 추가
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response))
    
    return response

# 대화 진행
print("\n👤 1턴: 나는 파이썬을 공부하고 있어.")
print(f"🤖 {chat_with_history('나는 파이썬을 공부하고 있어.')}")

print("\n" + "-" * 40)
print("\n👤 2턴: 어떤 프레임워크를 배우면 좋을까?")
print(f"🤖 {chat_with_history('어떤 프레임워크를 배우면 좋을까?')}")

print("\n" + "-" * 40)
print("\n👤 3턴: 내가 지금 뭘 공부하고 있다고 했지?")
print(f"🤖 {chat_with_history('내가 지금 뭘 공부하고 있다고 했지?')}")

print(f"\n📊 대화 이력: {len(chat_history)}개 메시지")
print("✅ 이전 대화 맥락을 기억하고 있습니다!")

💾 메시지 히스토리를 활용한 대화 관리

👤 1턴: 나는 파이썬을 공부하고 있어.
🤖 좋아요! 파이썬은 배우기 쉽고 강력한 프로그래밍 언어입니다. 어떤 부분을 공부하고 있나요? 혹시 도움이 필요하면 말씀해 주세요!

----------------------------------------

👤 2턴: 어떤 프레임워크를 배우면 좋을까?
🤖 파이썬에서는 다양한 프레임워크가 있습니다. 여러분의 관심 분야에 따라 추천할 수 있는 프레임워크는 다릅니다. 몇 가지 예를 들어볼게요:

1. **웹 개발**:
   - **Django**: 대규모 웹 애플리케이션을 만드는 데 적합하고, 많은 기능을 내장하고 있어 빠르게 개발할 수 있습니다.
   - **Flask**: 경량 프레임워크로, 유연성과 확장성이 뛰어나고 소규모 프로젝트에 적합합니다.

2. **데이터 과학 및 머신러닝**:
   - **Pandas**: 데이터 분석을 위한 라이브러리로, 데이터 조작과 분석에 유용합니다.
   - **NumPy**: 수치 계산을 위한 라이브러리로, 배열과 행렬 연산에 강력합니다.
   - **TensorFlow** 또는 **PyTorch**: 머신러닝 및 딥러닝 모델을 개발하는 데 많이 사용됩니다.

3. **GUI 애플리케이션 개발**:
   - **Tkinter**: 파이썬에 내장된 GUI 라이브러리로, 간단한 GUI 애플리케이션을 만드는 데 적합합니다.
   - **PyQt**: 보다 복잡하고 세련된 GUI 애플리케이션을 만들 수 있는 프레임워크입니다.

어떤 분야에 관심이 있는지에 따라 선택하면 좋습니다! 더 궁금한 점이 있으면 말씀해 주세요.

----------------------------------------

👤 3턴: 내가 지금 뭘 공부하고 있다고 했지?
🤖 당신은 지금 파이썬을 공부하고 있다고 말씀하셨습니다. 파이썬 공부에 도움이 필요하시거나 궁금한 점이 있다면 언제든지 말씀해 주세요!

📊 대화 이력: 6개 메시지
✅ 이전 대화 맥락을 기억하고 있습니다!


In [21]:
# 대화 이력 확인 및 관리 전략
print("=" * 60)
print("📋 대화 이력 확인")
print("=" * 60)

for i, msg in enumerate(chat_history):
    role = "👤 User" if isinstance(msg, HumanMessage) else "🤖 AI"
    content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
    print(f"\n[{i}] {role}: {content}")

print(f"\n{'=' * 60}")
print("💡 대화 이력 관리 전략:")
print("   1. 전체 보관: 짧은 대화에 적합")
print("   2. 윈도우: 최근 N턴만 유지 (메모리/비용 절약)")
print("   3. 요약: 오래된 대화를 요약하여 보관")
print("   4. 토큰 제한: 최대 토큰 수 기준으로 관리")

# 윈도우 방식 예시
MAX_HISTORY = 4  # 최근 4개 메시지만 유지 (2턴)
if len(chat_history) > MAX_HISTORY:
    trimmed = chat_history[-MAX_HISTORY:]
    print(f"\n📌 윈도우 적용 예시: {len(chat_history)}개 → {len(trimmed)}개")

📋 대화 이력 확인

[0] 👤 User: 나는 파이썬을 공부하고 있어.

[1] 🤖 AI: 좋아요! 파이썬은 배우기 쉽고 강력한 프로그래밍 언어입니다. 어떤 부분을 공부하고 있나요? 혹시 도움이 필요하면 말씀해 주세요!

[2] 👤 User: 어떤 프레임워크를 배우면 좋을까?

[3] 🤖 AI: 파이썬에서는 다양한 프레임워크가 있습니다. 여러분의 관심 분야에 따라 추천할 수 있는 프레임워크는 다릅니다. 몇 가지 예를 들어볼게요:

1. ...

[4] 👤 User: 내가 지금 뭘 공부하고 있다고 했지?

[5] 🤖 AI: 당신은 지금 파이썬을 공부하고 있다고 말씀하셨습니다. 파이썬 공부에 도움이 필요하시거나 궁금한 점이 있다면 언제든지 말씀해 주세요!

💡 대화 이력 관리 전략:
   1. 전체 보관: 짧은 대화에 적합
   2. 윈도우: 최근 N턴만 유지 (메모리/비용 절약)
   3. 요약: 오래된 대화를 요약하여 보관
   4. 토큰 제한: 최대 토큰 수 기준으로 관리

📌 윈도우 적용 예시: 6개 → 4개


---

## 7️⃣ 간단한 챗봇 구현

지금까지 배운 내용을 종합하여 **대화형 챗봇**을 구현합니다.

### 챗봇 구성요소

1. **시스템 프롬프트**: 챗봇의 역할 정의
2. **프롬프트 템플릿**: MessagesPlaceholder로 이력 관리
3. **LCEL 체인**: prompt | model | parser
4. **대화 이력**: 최근 N턴 윈도우 관리

---

In [22]:
# 간단한 챗봇 클래스 구현
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI

print("=" * 60)
print("🤖 간단한 챗봇 구현")
print("=" * 60)

class SimpleChatbot:
    """LangChain 기반 간단한 챗봇"""
    
    def __init__(self, system_prompt, model_name="gpt-4o-mini", 
                 temperature=0.7, max_history=10):
        self.history = []
        self.max_history = max_history
        
        # 프롬프트 템플릿
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}"),
        ])
        
        # 모델
        self.model = ChatOpenAI(
            model=model_name,
            temperature=temperature,
            max_tokens=500,
        )
        
        # 체인 구성
        self.chain = self.prompt | self.model | StrOutputParser()
        
        print(f"✅ 챗봇 초기화 완료!")
        print(f"   모델: {model_name}")
        print(f"   최대 이력: {max_history}개 메시지")
    
    def chat(self, user_input):
        """사용자 입력에 대한 응답 생성"""
        # 체인 실행
        response = self.chain.invoke({
            "history": self.history,
            "input": user_input,
        })
        
        # 이력 추가
        self.history.append(HumanMessage(content=user_input))
        self.history.append(AIMessage(content=response))
        
        # 이력 크기 관리 (윈도우)
        if len(self.history) > self.max_history:
            self.history = self.history[-self.max_history:]
        
        return response
    
    def stream_chat(self, user_input):
        """스트리밍 응답 생성"""
        full_response = ""
        for chunk in self.chain.stream({"history": self.history, "input": user_input}):
            full_response += chunk
            yield chunk
        
        # 이력 추가
        self.history.append(HumanMessage(content=user_input))
        self.history.append(AIMessage(content=full_response))
        
        if len(self.history) > self.max_history:
            self.history = self.history[-self.max_history:]
    
    def reset(self):
        """대화 이력 초기화"""
        self.history = []
        print("🔄 대화 이력이 초기화되었습니다.")
    
    def get_history_count(self):
        """현재 대화 이력 수"""
        return len(self.history)

# 챗봇 생성
bot = SimpleChatbot(
    system_prompt="""당신은 AI/ML 학습을 도와주는 친절한 튜터입니다.
규칙:
- 한국어로 답변합니다.
- 초보자도 이해할 수 있게 쉽게 설명합니다.
- 필요하면 예시 코드를 포함합니다.
- 답변은 간결하게 유지합니다.""",
    max_history=10,
)

🤖 간단한 챗봇 구현
✅ 챗봇 초기화 완료!
   모델: gpt-4o-mini
   최대 이력: 10개 메시지


In [23]:
# 챗봇과 대화하기
print("=" * 60)
print("💬 챗봇과 대화하기")
print("=" * 60)

# 대화 1
print("\n👤 User: LoRA가 뭐야?")
print(f"🤖 Bot: {bot.chat('LoRA가 뭐야?')}")

# 대화 2
print(f"\n{'─' * 50}")
print("\n👤 User: 그거랑 QLoRA는 뭐가 달라?")
print(f"🤖 Bot: {bot.chat('그거랑 QLoRA는 뭐가 달라?')}")

# 대화 3 (스트리밍)
print(f"\n{'─' * 50}")
print("\n👤 User: 내 GPU가 8GB인데 어떤 모델까지 학습 가능해?")
print("🤖 Bot: ", end="")
for chunk in bot.stream_chat("내 GPU가 8GB인데 어떤 모델까지 학습 가능해?"):
    print(chunk, end="", flush=True)
print()

print(f"\n📊 현재 대화 이력: {bot.get_history_count()}개 메시지")

💬 챗봇과 대화하기

👤 User: LoRA가 뭐야?
🤖 Bot: LoRA는 "Low-Rank Adaptation"의 약자로, 주로 딥러닝 모델을 효율적으로 학습하고 조정하기 위해 사용되는 기술입니다. LoRA는 기존의 큰 모델을 사용할 때, 전체 모델을 다시 훈련하는 대신 소규모의 파라미터(저순위 행렬)를 추가하여 필요한 조정을 하는 방법입니다.

### 왜 LoRA를 사용할까요?
1. **효율성**: 전체 모델을 훈련하는 것보다 적은 자원으로 빠르게 조정할 수 있습니다.
2. **메모리 절약**: 추가적인 파라미터만 학습하므로 메모리 사용량이 적습니다.
3. **전이 학습**: 이미 훈련된 모델을 기반으로 새로운 작업에 적합하도록 쉽게 조정할 수 있습니다.

### 간단한 예시
LoRA를 사용하여 모델을 조정하는 경우를 생각해봅시다.

```python
import torch
import torch.nn as nn

# 기존 모델 정의
class OriginalModel(nn.Module):
    def __init__(self):
        super(OriginalModel, self).__init__()
        self.fc = nn.Linear(10, 5)

    def forward(self, x):
        return self.fc(x)

# LoRA 레이어 추가
class LoRALayer(nn.Module):
    def __init__(self, input_dim, output_dim, rank):
        super(LoRALayer, self).__init__()
        self.A = nn.Parameter(torch.randn(input_dim, rank))
        self.B = nn.Parameter(torch.randn(rank, output_dim))

    def forward(self, x):
        return x @ self.A @ self.B

# 전체 모델 조정
c

In [24]:
# 대화 맥락 유지 확인
print("=" * 60)
print("🔄 대화 맥락 유지 확인")
print("=" * 60)

print("\n👤 User: 앞에서 설명해준 내용을 요약해줘.")
print(f"🤖 Bot: {bot.chat('앞에서 설명해준 내용을 요약해줘.')}")

print(f"\n📊 대화 이력: {bot.get_history_count()}개 메시지")
print("\n✅ 이전 대화 맥락을 기억하고 요약할 수 있습니다!")

# 대화 초기화
bot.reset()

🔄 대화 맥락 유지 확인

👤 User: 앞에서 설명해준 내용을 요약해줘.
🤖 Bot: 8GB GPU에서 학습할 수 있는 모델은 여러 요인에 따라 달라집니다:

1. **모델 크기**: 작은 모델(예: MobileNet, ResNet-18)은 원활하게 학습 가능, 큰 모델(예: GPT-3, BERT-large)은 어려울 수 있음.
   
2. **배치 크기**: 배치 크기가 클수록 메모리 사용량이 증가하므로, 배치 크기를 줄이면 더 큰 모델을 사용할 수 있음.

3. **데이터셋 크기**: 데이터셋의 크기와 복잡성도 메모리 사용에 영향을 미침.

4. **혼합 정밀도 훈련**: 16비트 부동소수점을 사용하면 메모리 사용량을 줄이고 더 큰 모델 학습 가능.

### 결론
8GB GPU에서는 작은에서 중간 크기의 모델을 학습하는 것이 적합하며, 메모리 사용량을 모니터링하면서 적절한 배치 크기를 조절하는 것이 중요합니다.

📊 대화 이력: 8개 메시지

✅ 이전 대화 맥락을 기억하고 요약할 수 있습니다!
🔄 대화 이력이 초기화되었습니다.


In [25]:
# 🎮 인터랙티브 챗봇 (직접 대화해보기)
import sys

print("=" * 60)
print("🎮 인터랙티브 챗봇 — 직접 대화해보세요!")
print("=" * 60)
print("💡 'quit' 또는 'q'를 입력하면 종료합니다.")
print("💡 'reset'을 입력하면 대화 이력을 초기화합니다.")
print("💡 'history'를 입력하면 현재 대화 이력을 확인합니다.")
print()

# 위에서 만든 bot을 그대로 사용 (이력 초기화)
bot.reset()

while True:
    user_input = input("\n👤 You: ").strip()
    
    if not user_input:
        continue
    
    if user_input.lower() in ['quit', 'q', '종료']:
        print("\n👋 대화를 종료합니다.")
        print(f"📊 총 대화 이력: {bot.get_history_count()}개 메시지")
        break
    
    if user_input.lower() == 'reset':
        bot.reset()
        continue
    
    if user_input.lower() == 'history':
        print(f"\n📊 현재 대화 이력: {bot.get_history_count()}개 메시지")
        for i, msg in enumerate(bot.history):
            role = "👤" if isinstance(msg, HumanMessage) else "🤖"
            content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
            print(f"   {role} [{i}] {content}")
        sys.stdout.flush()
        continue
    
    # 일반 응답 (Jupyter input() 버퍼링 문제 방지)
    response = bot.chat(user_input)
    print(f"🤖 Bot: {response}")
    sys.stdout.flush()


🎮 인터랙티브 챗봇 — 직접 대화해보세요!
💡 'quit' 또는 'q'를 입력하면 종료합니다.
💡 'reset'을 입력하면 대화 이력을 초기화합니다.
💡 'history'를 입력하면 현재 대화 이력을 확인합니다.

🔄 대화 이력이 초기화되었습니다.
🤖 Bot: 안녕하세요! AI/ML에 대해 궁금한 점이 있으신가요? 어떤 도움을 드릴 수 있을지 말씀해 주세요!
🤖 Bot: AI(인공지능)의 전망은 매우 밝습니다. 몇 가지 주요 포인트를 소개할게요.

1. **산업 혁신**: AI는 제조, 의료, 금융 등 다양한 산업에서 효율성을 높이고 비용을 절감하는 데 기여하고 있습니다. 예를 들어, AI를 이용한 데이터 분석으로 더 빠르고 정확한 의사 결정을 할 수 있습니다.

2. **자율주행차**: 자율주행 기술은 AI의 발전을 통해 더욱 현실화되고 있습니다. 앞으로는 더 많은 차량이 AI를 이용해 안전하게 운전할 수 있을 것입니다.

3. **개인화 서비스**: AI는 사용자 데이터를 분석하여 개인 맞춤형 추천 서비스를 제공합니다. 예를 들어, 넷플릭스나 아마존에서 여러분의 취향에 맞는 콘텐츠나 상품을 추천하는 것이죠.

4. **AI와 인간의 협업**: AI는 인간의 업무를 보조하거나 반복적인 작업을 자동화하여 사람들이 더 창의적인 일에 집중할 수 있도록 도와줍니다.

5. **윤리적 문제**: AI의 발전과 함께 개인정보 보호, 편향 문제 등 윤리적 이슈도 중요해지고 있습니다. 이에 대한 논의와 해결책이 필요합니다.

결론적으로, AI는 앞으로도 다양한 분야에서 큰 변화를 가져올 것으로 예상되며, 기술의 발전과 함께 사회적 책임도 중요해질 것입니다. 궁금한 점이 더 있으신가요?

👋 대화를 종료합니다.
📊 총 대화 이력: 4개 메시지


---

## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

| 주제 | 핵심 내용 |
|------|----------|
| OpenAI API 직접 호출 | `client.chat.completions.create(...)`, role 구조, 스트리밍 |
| LangChain 구조 | langchain-core, langchain-openai 등 모듈화된 구조 |
| ChatModel | ChatOpenAI로 모델 생성, invoke/stream/batch 지원 |
| PromptTemplate | ChatPromptTemplate으로 재사용 가능한 프롬프트 관리 |
| LCEL | `|` 연산자로 prompt → model → parser 파이프라인 구성 |
| Output Parser | StrOutputParser, JsonOutputParser 등으로 출력 구조화 |
| Memory | MessagesPlaceholder + 대화 이력 리스트로 맥락 유지 |
| 챗봇 | 위 요소들을 조합하여 대화형 AI 구현 |

### OpenAI API → LangChain 매핑

```python
# 원본 OpenAI API
client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "..."}],
    temperature=0.7,
)

# LangChain (같은 동작을 추상화)
ChatOpenAI(model="gpt-4o-mini", temperature=0.7).invoke([HumanMessage("...")])
```

### LCEL 핵심 정리

```python
chain = prompt | model | parser
chain.invoke({...})   # 단일 실행
chain.stream({...})   # 스트리밍
chain.batch([...])    # 배치 처리
```

### 다음 세션 예고

- 🔜 **Session 7**: microGPT 실습 - Karpathy의 의존성 없는 243줄 GPT

---

### 💡 실습 과제

1. OpenAI API를 직접 호출하여 자신의 시스템 프롬프트(예: "초등학교 과학 선생님")를 적용해보세요.
2. 같은 질문을 OpenAI API 직접 호출과 LangChain `ChatOpenAI` 두 방식으로 호출해 결과/코드를 비교해보세요.
3. SimpleChatbot 클래스를 활용하여 자신만의 전문 분야 챗봇을 만들어보세요.
4. LCEL 파이프라인을 확장하여 '질문 → 답변 → 영어 번역' 3단계 체인을 만들어보세요.
5. JsonOutputParser를 활용하여 뉴스 기사에서 '제목, 날짜, 핵심 키워드, 요약'을 추출하는 체인을 만들어보세요.

---